<a href="https://colab.research.google.com/github/nrocka/SD-Anime-Prompt-to-Danbooru-Tags/blob/main/Anime_SD_Prompt_to_Danbooru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I am not a programmer at all, so im lacking a lot of knowledge. This thing was built entirely using a Chatbot AI. You will probably notice. The cells do work though (as far as i was able to test)

## 🟥🟥🟥 **Preperation** 🟥🟥🟥

In [2]:
# Import Cell
# @title 0️⃣ Import Dependencies
# @markdown This cell contains all necessary imports.
import re
import requests
import pandas as pd
from IPython.display import display, HTML, Javascript

In [3]:
# Global Parameters Cell
# @title ✨ Global Parameters used throughout the notebook
# @markdown This parameter defines the maximum number of characters allowed before inserting a newline. Setting this to 0 will disable these static newlines.
newline_threshold = 0 # @param {type:"integer"}

# @markdown This parameter decides whether a newline should cut off only after completing a tag or directly after reaching the threshold.
only_cut_after_complete_tags = False # @param {type:"boolean"}

## 🟩🟩🟩 **Prompt Editor** 🟩🟩🟩

In [4]:
# First Clean-up Cell: Remove LoRAs, Weights, and Brackets
# @title 1️⃣ Clean Up Prompt
# @markdown This cell removes LoRA triggers (`<...>`), weighted tags like `(tag:1.2)`, and unescaped brackets.

raw_prompt_input_1 = ""  # @param {type:"string"}

def clean_prompt_step1(input_prompt):
    # 1. Remove tags like <lora:...>
    prompt_no_chevrons = re.sub(r'<.*?>', '', input_prompt)

    # 2. Remove weights (e.g., (tag:1.2) → (tag))
    prompt_no_weights = re.sub(r'\(([^:()]+):[^()]*\)', r'(\1)', prompt_no_chevrons)

    # 3. Remove all brackets unless escaped: ( → '', ) → ''
    prompt_no_brackets = re.sub(r'(?<!\\)[()]', '', prompt_no_weights)

    # 4. Remove excessive spaces
    prompt_single_spaced = re.sub(r'\s{2,}', ' ', prompt_no_brackets)

    return prompt_single_spaced.strip()

cleaned_prompt_step1 = clean_prompt_step1(raw_prompt_input_1)

# Format for readability if needed
if newline_threshold == 0:
    formatted_output_step1 = cleaned_prompt_step1
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in cleaned_prompt_step1.split(', '):
        if len(current) + len(tag) + 2 <= newline_threshold:
            current += (', ' if current else '') + tag
        else:
            lines.append(current.rstrip(', '))
            current = tag
    if current:
        lines.append(current.rstrip(', '))
    formatted_output_step1 = '\n'.join(lines)
else:
    formatted_output_step1 = '\n'.join(
        [cleaned_prompt_step1[i:i + newline_threshold] for i in range(0, len(cleaned_prompt_step1), newline_threshold)]
    )

# Static textarea display
display(HTML(f"""
    <textarea id="outputText1" rows="10" style="width:50%;">{formatted_output_step1}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText1').value)">Copy to Clipboard</button>
"""))






---




In [5]:
# Second Clean-up Cell: Remove Quality, Source, and Score Tags
# @title 2️⃣ Remove Quality, Source, and Score Tags
# @markdown This cell removes `score_`, `source_`, and known quality/date tags. You can also optionally remove custom tags.

# Raw input for this step
raw_prompt_input_2 = cleaned_prompt_step1  # Pass output from step 1

# @markdown Checking this box will allow you to remove additional tags (comma-separated).
remove_additional_tags = False  # @param {type:"boolean"}
additional_tags = ""  # @param {type:"string"}

def clean_prompt_step2(input_prompt, remove_extra=False, extra_tags=""):
    # 1. Remove all score_ and source_ prompts
    prompt_no_prefixes = re.sub(r'\b(score_|source_)[^,\s]*[\s,]*', '', input_prompt)

    # 2. Remove known quality-related tags
    quality_tags = ["masterpiece", "best quality", "low quality", "amazing quality"]
    for tag in quality_tags:
        prompt_no_prefixes = prompt_no_prefixes.replace(tag, '')

    # 3. Remove common vague time/date tags
    date_tags = ["old", "early", "mid", "recent", "newest"]
    for tag in date_tags:
        prompt_no_prefixes = prompt_no_prefixes.replace(tag, '')

    # 4. Remove user-defined additional tags
    if remove_extra and extra_tags.strip():
        custom_tags = [t.strip() for t in extra_tags.split(',')]
        for tag in custom_tags:
            prompt_no_prefixes = prompt_no_prefixes.replace(tag, '')

    # Clean up spaces and delimiters
    prompt_cleaned = re.sub(r',\s*', ', ', prompt_no_prefixes).strip(', ')
    prompt_cleaned = re.sub(r',\s*,+', ', ', prompt_cleaned)
    prompt_cleaned = re.sub(r',\s*,', ',', prompt_cleaned)
    prompt_cleaned = re.sub(r'\s+', ' ', prompt_cleaned).strip()

    return prompt_cleaned

cleaned_prompt_step2 = clean_prompt_step2(raw_prompt_input_2, remove_additional_tags, additional_tags)

# Format for readability
if newline_threshold == 0:
    formatted_output_step2 = cleaned_prompt_step2
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in cleaned_prompt_step2.split(', '):
        if len(current) + len(tag) + 2 <= newline_threshold:
            current += (', ' if current else '') + tag
        else:
            lines.append(current.rstrip(', '))
            current = tag
    if current:
        lines.append(current.rstrip(', '))
    formatted_output_step2 = '\n'.join(lines)
else:
    formatted_output_step2 = '\n'.join(
        [cleaned_prompt_step2[i:i + newline_threshold] for i in range(0, len(cleaned_prompt_step2), newline_threshold)]
    )

# Display
display(HTML(f"""
    <textarea id="outputText2" rows="10" style="width:50%;">{formatted_output_step2}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText2').value)">Copy to Clipboard</button>
"""))




---



In [6]:
# Third Clean-up Cell: Format to Danbooru Input
# @title 3️⃣ Format to Danbooru Input
# @markdown This cell replaces spaces in tags with underscores, removes commas, and removes backslashes (\\).

# Raw input for this step
raw_prompt_input_3 = cleaned_prompt_step2  # Pass output from step 2

def clean_prompt_step3(input_prompt):
    # 1. Replace spaces within each tag with underscores
    tags = input_prompt.split(', ')
    tags = [tag.strip().replace(' ', '_') for tag in tags]

    # 2. Remove backslashes
    tags = [tag.replace('\\', '') for tag in tags]

    # 3. Remove commas and join into space-separated list
    return ' '.join(tags)

cleaned_prompt_step3 = clean_prompt_step3(raw_prompt_input_3)

# Format for readability
if newline_threshold == 0:
    formatted_output_step3 = cleaned_prompt_step3
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in cleaned_prompt_step3.split(' '):  # Now space-separated, not comma
        if len(current) + len(tag) + 1 <= newline_threshold:
            current += (' ' if current else '') + tag
        else:
            lines.append(current)
            current = tag
    if current:
        lines.append(current)
    formatted_output_step3 = '\n'.join(lines)
else:
    formatted_output_step3 = '\n'.join(
        [cleaned_prompt_step3[i:i + newline_threshold] for i in range(0, len(cleaned_prompt_step3), newline_threshold)]
    )

# Display
display(HTML(f"""
    <textarea id="outputText3" rows="10" style="width:50%;">{formatted_output_step3}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText3').value)">Copy to Clipboard</button>
"""))




---



In [8]:
# Fourth Clean-up Cell: List Unsupported and Unknown Tags
# @title 4️⃣ List Unsupported and Unknown Tags (CSV or Danbooru Lookup)
# @markdown This cell lists unsupported or unknown tags using one or more sources. You can select multiple validation methods below.

# @markdown ✅ Choose which validation sources to use:
use_csv_validation = False  # @param {type:"boolean"}
use_danbooru_lookup = True  # @param {type:"boolean"}

# @markdown ✅ Choose which output(s) to show:
show_unsupported_tags = True  # @param {type:"boolean"}
show_supported_tags = True  # @param {type:"boolean"}

# @markdown CSV URLs (comma-separated) if CSV validation is used
csv_urls = "https://huggingface.co/SmilingWolf/wd-vit-large-tagger-v3/raw/main/selected_tags.csv"  # @param {type:"string"}

raw_prompt_input_4 = cleaned_prompt_step3  # Uses Danbooru-formatted prompt from previous step

import requests
import pandas as pd

def check_tags(formatted_prompt, use_csv, use_lookup, csv_links):
    tags = formatted_prompt.split(' ')
    supported = set()
    unsupported = set(tags)  # Start assuming all unsupported

    # CSV validation
    if use_csv:
        csv_sources = [url.strip() for url in csv_links.split(',') if url.strip()]
        all_csv_tags = set()
        for url in csv_sources:
            try:
                csv_tags = pd.read_csv(url)['name'].tolist()
                all_csv_tags.update(tag.strip() for tag in csv_tags)
            except Exception as e:
                print(f"⚠️ Failed to load CSV from {url}: {e}")
        supported.update(set(tags) & all_csv_tags)

    # Danbooru tag lookup validation
    if use_lookup:
        def is_tag_supported(tag):
            resp = requests.get(f"https://danbooru.donmai.us/tags.json?search[name_matches]={tag}")
            return resp.status_code == 200 and bool(resp.json())
        for tag in tags:
            if is_tag_supported(tag):
                supported.add(tag)

    unsupported = set(tags) - supported
    return sorted(list(supported)), sorted(list(unsupported))

# Check for empty input
if not raw_prompt_input_4.strip():
    print("⚠️ No tag input detected from the previous cell. Please run earlier steps first.")
else:
    supported_tags, unsupported_tags = check_tags(
        raw_prompt_input_4,
        use_csv_validation,
        use_danbooru_lookup,
        csv_urls
    )

    # Display unsupported tags
    if show_unsupported_tags and unsupported_tags:
        formatted_unsupported = ' '.join(unsupported_tags)
        if newline_threshold == 0:
            formatted_unsupported = formatted_unsupported
        elif only_cut_after_complete_tags:
            lines = []
            current = ""
            for tag in unsupported_tags:
                if len(current) + len(tag) + 1 <= newline_threshold:
                    current += (' ' if current else '') + tag
                else:
                    lines.append(current)
                    current = tag
            if current:
                lines.append(current)
            formatted_unsupported = '\n'.join(lines)
        else:
            formatted_unsupported = '\n'.join(
                [formatted_unsupported[i:i + newline_threshold] for i in range(0, len(formatted_unsupported), max(1, newline_threshold))]
            )

        display(HTML(f"""
            <h4>❌ Unsupported / Unknown Tags</h4>
            <textarea id="outputText4_unsupported" rows="10" style="width:50%; background:#1e1e1e; color:white; border:1px solid #555;">{formatted_unsupported}</textarea><br>
            <button onclick="navigator.clipboard.writeText(document.getElementById('outputText4_unsupported').value)">Copy to Clipboard</button>
        """))

    elif show_unsupported_tags:
        print("✅ No unsupported tags found.")

    # Display supported tags
    if show_supported_tags and supported_tags:
        formatted_supported = ' '.join(supported_tags)
        if newline_threshold == 0:
            formatted_supported = formatted_supported
        elif only_cut_after_complete_tags:
            lines = []
            current = ""
            for tag in supported_tags:
                if len(current) + len(tag) + 1 <= newline_threshold:
                    current += (' ' if current else '') + tag
                else:
                    lines.append(current)
                    current = tag
            if current:
                lines.append(current)
            formatted_supported = '\n'.join(lines)
        else:
            formatted_supported = '\n'.join(
                [formatted_supported[i:i + newline_threshold] for i in range(0, len(formatted_supported), max(1, newline_threshold))]
            )

        display(HTML(f"""
            <h4>✅ Supported Tags</h4>
            <textarea id="outputText4_supported" rows="10" style="width:50%; background:#1e1e1e; color:white; border:1px solid #555;">{formatted_supported}</textarea><br>
            <button onclick="navigator.clipboard.writeText(document.getElementById('outputText4_supported').value)">Copy to Clipboard</button>
        """))

    elif show_supported_tags:
        print("⚠️ No supported tags detected.")


⚠️ No tag input detected from the previous cell. Please run earlier steps first.




---



## ✨✨✨ **Extras & Curate Tags** ✨✨✨

In [9]:
# Add Tags to Danbooru Tags Cell
# @title ➕ Add Tags to Danbooru Tags
# @markdown This cell accepts a Danbooru tag list from the user and allows them to add additional tags.

# @markdown ✅ Check this box to reuse the previous result (final_prompt) as the input for this run. It won't update the input_tags field visually but works in the background.
reuse_result = True  # @param {type:"boolean"}

# @markdown 💬 Enter your Danbooru tag list here:
if not reuse_result:
    input_tags = ""  # @param {type:"string"}

# @markdown ➕ Additional tags to append to the list (space-separated):
additional_tags = ""  # @param {type:"string"}

# Use previous result if enabled
if reuse_result:
    try:
        input_tags = final_prompt
    except NameError:
        input_tags = ""  # Fallback in case final_prompt doesn't exist yet

def add_tags(input_str, extra_str):
    base_tags = input_str.strip().split(' ') if input_str.strip() else []
    added_tags = extra_str.strip().split(' ') if extra_str.strip() else []

    # Add only if not already present, preserving order
    final = base_tags + [tag for tag in added_tags if tag and tag not in base_tags]
    return ' '.join(final)

# Generate final prompt
final_prompt = add_tags(input_tags, additional_tags)

# Format output
if newline_threshold == 0:
    formatted_output_add = final_prompt
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in final_prompt.split(' '):
        if len(current) + len(tag) + 1 <= newline_threshold:
            current += (' ' if current else '') + tag
        else:
            lines.append(current)
            current = tag
    if current:
        lines.append(current)
    formatted_output_add = '\n'.join(lines)
else:
    formatted_output_add = '\n'.join(
        [final_prompt[i:i + newline_threshold] for i in range(0, len(final_prompt), newline_threshold)]
    )

# Display result
display(HTML(f"""
    <h4>✅ Final Prompt After Adding Tags</h4>
    <textarea id="outputText5" rows="10" style="width:50%;">{formatted_output_add}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText5').value)">Copy to Clipboard</button>
"""))




---



In [12]:
# Remove Tags from Danbooru Tags Cell
# @title ➖ Remove Tags from Danbooru Tags
# @markdown This cell accepts a Danbooru tag list from the user and allows them to remove specific tags.

# @markdown ✅ Check this box to reuse the previous result (final_prompt) as the input for this run. It won't update the input_tags field visually but works in the background.
reuse_result = True  # @param {type:"boolean"}

# @markdown 💬 Enter your Danbooru tag list here:
if not reuse_result:
    input_tags = ""  # @param {type:"string"}

# @markdown ➖ Tags to remove (space-separated):
tags_to_remove = "good best hires-upscaled non-web_source"  # @param {type:"string"}

# Use previous result if enabled
if reuse_result:
    try:
        input_tags = final_prompt
    except NameError:
        input_tags = ""

def remove_tags(input_str, remove_str):
    base_tags = input_str.strip().split(' ') if input_str.strip() else []
    remove_list = remove_str.strip().split(' ') if remove_str.strip() else []

    final = [tag for tag in base_tags if tag and tag not in remove_list]
    return ' '.join(final)

# Generate final prompt
final_prompt = remove_tags(input_tags, tags_to_remove)

# Format output
if newline_threshold == 0:
    formatted_output_remove = final_prompt
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in final_prompt.split(' '):
        if len(current) + len(tag) + 1 <= newline_threshold:
            current += (' ' if current else '') + tag
        else:
            lines.append(current)
            current = tag
    if current:
        lines.append(current)
    formatted_output_remove = '\n'.join(lines)
else:
    formatted_output_remove = '\n'.join(
        [final_prompt[i:i + newline_threshold] for i in range(0, len(final_prompt), newline_threshold)]
    )

# Display result
display(HTML(f"""
    <h4>🗑️ Final Prompt After Removing Tags</h4>
    <textarea id="outputText6" rows="10" style="width:50%;">{formatted_output_remove}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText6').value)">Copy to Clipboard</button>
"""))




---



In [13]:
# Sort Tags Alphabetically Cell
# @title 🔤 Sort Tags Alphabetically
# @markdown This cell allows you to sort the tags alphabetically.

# @markdown ✅ Check this box to reuse the previous result (final_prompt) as the input for this run. It won't update the input_tags field visually but works in the background.
reuse_result = False  # @param {type:"boolean"}

# @markdown 💬 Enter your Danbooru tag list here:
if not reuse_result:
    input_tags = ""  # @param {type:"string"}

# Use previous result if enabled
if reuse_result:
    try:
        input_tags = final_prompt
    except NameError:
        input_tags = ""

def sort_tags_alphabetically(tags_str):
    tags = tags_str.strip().split(' ') if tags_str.strip() else []
    sorted_tags = sorted([tag for tag in tags if tag])
    return ' '.join(sorted_tags)

# Generate sorted prompt
final_prompt = sort_tags_alphabetically(input_tags)

# Format output
if newline_threshold == 0:
    formatted_output_sort = final_prompt
elif only_cut_after_complete_tags:
    lines = []
    current = ""
    for tag in final_prompt.split(' '):
        if len(current) + len(tag) + 1 <= newline_threshold:
            current += (' ' if current else '') + tag
        else:
            lines.append(current)
            current = tag
    if current:
        lines.append(current)
    formatted_output_sort = '\n'.join(lines)
else:
    formatted_output_sort = '\n'.join(
        [final_prompt[i:i + newline_threshold] for i in range(0, len(final_prompt), newline_threshold)]
    )

# Display result
display(HTML(f"""
    <h4>🔠 Final Prompt After Sorting Alphabetically</h4>
    <textarea id="outputText7" rows="10" style="width:50%;">{formatted_output_sort}</textarea><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('outputText7').value)">Copy to Clipboard</button>
"""))




---



In [14]:
# Clear All Inputs and Outputs Cell
# @title ✨ Clear All Inputs and Outputs
# @markdown This cell resets all relevant inputs and outputs to a clean state.

def clear_all():
    # Declare each global variable explicitly
    global raw_prompt_input_1, cleaned_prompt_step1, formatted_output_step1
    global raw_prompt_input_2, cleaned_prompt_step2, formatted_output_step2
    global raw_prompt_input_3, cleaned_prompt_step3, formatted_output_step3
    global raw_prompt_input_4, supported_tags, unsupported_tags
    global final_prompt, input_tags, additional_tags, tags_to_remove
    global formatted_output_add, formatted_output_remove, formatted_output_sort
    global reuse_result, csv_urls

    # Reset prompt steps
    raw_prompt_input_1 = ""
    cleaned_prompt_step1 = ""
    formatted_output_step1 = ""

    raw_prompt_input_2 = ""
    cleaned_prompt_step2 = ""
    formatted_output_step2 = ""

    raw_prompt_input_3 = ""
    cleaned_prompt_step3 = ""
    formatted_output_step3 = ""

    raw_prompt_input_4 = ""
    supported_tags = []
    unsupported_tags = []

    # Final tag state
    final_prompt = ""
    input_tags = ""
    additional_tags = ""
    tags_to_remove = ""

    formatted_output_add = ""
    formatted_output_remove = ""
    formatted_output_sort = ""

    reuse_result = False
    csv_urls = ""

    print("✅ All inputs and outputs have been cleared.")

clear_all()

✅ All inputs and outputs have been cleared.
